# Blood Cell Classification — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s5-blood-cells.ipynb)

Classify microscopy images of **blood cells** into one of **8 cell types**, from
28×28 RGB images: [challenge 173](https://ml-arena.com/viewchallenge/173) on ML-Arena.

This notebook runs end-to-end: **download the data → look at it → train a simple model → submit**.
Metric: **F1-macro**.

> Open in Colab, then run the cells top to bottom. The only thing you must edit is
> your API token in the next cell (find it on your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk>=3" scikit-learn pandas numpy  # installs the `mlarena` package

import mlarena
import numpy as np
import pandas as pd

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 173

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`download_dataset` pulls the challenge files into `data/`. Each image is a
`28 × 28 × 3` (RGB) array; the training labels are in `y_train.csv` (`image_id,label`).

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")

train = np.load("data/train_images.npz")   # train["image_id"], train["images"] (N,28,28,3) uint8
test  = np.load("data/test_images.npz")    # test["image_id"],  test["images"]
y_train = pd.read_csv("data/y_train.csv")  # image_id,label

# line up each image with its label (match on image_id, don't assume row order)
label_of = y_train.set_index("image_id")["label"]
y = label_of.loc[train["image_id"]].values

print("train images:", train["images"].shape, "  test images:", test["images"].shape)
print(pd.Series(y).value_counts())

## 2. Look at the images

Six random training images per class, one row per cell type, with the number of
training images of that class. The rows differ in nucleus shape, granularity and
stain colour; the counts show how unbalanced the classes are, which is why the
metric is F1-macro.


In [ ]:
import matplotlib.pyplot as plt

classes = sorted(np.unique(y))
rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(classes), 6, figsize=(6, len(classes)))
for row, cls in zip(axes, classes):
    idx = rng.choice(np.flatnonzero(y == cls), size=6, replace=False)
    for ax, i in zip(row, idx):
        ax.imshow(train["images"][i])
        ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(f"{cls}\n({(y == cls).sum()})", rotation=0, ha="right", va="center")
plt.tight_layout()
plt.show()


## 3. A baseline model

Flatten each image into a vector of pixels and fit a random forest. This is just a
starting point — a small CNN (e.g. PyTorch) trained on the raw images will score
much higher. We hold out 20% to estimate the F1-macro before submitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X = train["images"].reshape(len(train["images"]), -1) / 255.0   # (N, 28*28*3)
X_test = test["images"].reshape(len(test["images"]), -1) / 255.0

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
clf.fit(X_tr, y_tr)
print("validation F1-macro: %.4f" % f1_score(y_val, clf.predict(X_val), average="macro"))

clf.fit(X, y)   # refit on all the training data before predicting the test set

## 4. Predict and write `submission.csv`

In [ ]:
pred = clf.predict(X_test)
submission = pd.DataFrame({"image_id": test["image_id"], "label": pred})
submission.to_csv("submission.csv", index=False)
submission.head()

## 5. Submit

Run the cell below to submit through the SDK, or upload `submission.csv` on the
challenge page.

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"])
print(result)

## 6. Where to go from here

The pixel-flattening baseline throws away the *spatial* structure of the image — that's
where most of the score is hiding. Work one step at a time: **change one thing → watch the
validation F1 → submit only if it improved.** Trust the held-out number, not the train score.

- **Look at more images.** Rerun the grid of section 2 with other seeds and look for anything odd — blurry crops, near-duplicates,
  mislabeled cells. F1-macro weights every class equally, so the rare ones matter most.

- **Train a small CNN (PyTorch).** A couple of conv + pooling layers already beats the random
  forest by a wide margin, because convolutions exploit the 2-D layout the baseline discards.
  Normalize inputs to `[0, 1]` (or per-channel mean/std) and use cross-entropy loss.

- **Train carefully.** Keep a stratified validation split and watch it every epoch.
  - *Learning rate* is the single most important knob — try a few values (e.g. 1e-2 … 1e-4).
  - Use **early stopping** and save the best-validation checkpoint.
  - Tune batch size and epochs; add dropout or weight decay if you overfit.

- **Augment the data.** Random flips, small rotations, and mild color jitter make the model
  robust to orientation and staining differences — cheap and almost always helps on microscopy.

- **Use a pretrained model.** Load a network pretrained on ImageNet (ResNet, EfficientNet, …),
  resize the 28×28 crops to its expected input, and **fine-tune** it on these cells. This is
  usually the biggest jump for the least effort.

Watch train vs. validation: if validation stalls while train keeps improving, you're
overfitting — stop early, augment more, or regularize.
